In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Cargar los datos
# Asegúrate de que los archivos train.csv y unique_m.csv estén en tu carpeta
train_df = pd.read_csv('train.csv')
unique_m_df = pd.read_csv('unique_m.csv')

# 2. Separar Predictores (X) y Variable Objetivo (y)
# Según el documento, train.csv tiene 81 variables + critical_temp
X = train_df.drop(columns=['critical_temp'])
y = train_df['critical_temp']

# 3. Estandarizar los predictores (Sección 3.4 del documento base)
# Ridge, Lasso y los métodos de selección son sensibles a la escala.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

# 4. División Train/Test (para evaluar el error de prueba)
# Usamos un 80/20. IMPORTANTE: El documento menciona que hay materiales repetidos.
# Para un análisis riguroso, se debería usar GroupShuffleSplit con la columna 'material' de unique_m.csv
# Pero para empezar, haremos una división aleatoria simple.
X_train, X_test, y_train, y_test = train_test_split(X_scaled_df, y, test_size=0.2, random_state=42)

print(f"Dimensiones de X_train: {X_train.shape}")
print(f"Dimensiones de y_train: {y_train.shape}")
print(f"Dimensiones de X_test: {X_test.shape}")
print(f"Dimensiones de y_test: {y_test.shape}")

Dimensiones de X_train: (17010, 81)
Dimensiones de y_train: (17010,)
Dimensiones de X_test: (4253, 81)
Dimensiones de y_test: (4253,)


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import time
import numpy as np
import pandas as pd

def forward_stepwise_selection(X, y, feature_names, max_features=20):
    """
    Realiza Forward Stepwise Selection.
    X: DataFrame o array de NumPy con los predictores.
    y: Serie o array con la variable objetivo.
    feature_names: Lista con los nombres de las columnas (para reportar).
    max_features: Número máximo de variables a seleccionar.
    """
    # Convertir X a NumPy para evitar problemas con nombres de columnas
    X_array = X.values if hasattr(X, 'values') else X
    y_array = y.values if hasattr(y, 'values') else y

    n_samples = X_array.shape[0]
    n_features = X_array.shape[1]

    remaining_indices = list(range(n_features))
    selected_indices = []
    current_rss = np.inf
    history = []

    for i in range(max_features):
        best_rss = np.inf
        best_index = None

        # Probar añadir cada característica restante
        for idx in remaining_indices:
            features_to_test = selected_indices + [idx]
            model = LinearRegression()
            model.fit(X_array[:, features_to_test], y_array)
            y_pred = model.predict(X_array[:, features_to_test])
            rss = mean_squared_error(y_array, y_pred) * n_samples

            if rss < best_rss:
                best_rss = rss
                best_index = idx

        # Si el RSS no mejora, detener
        if best_rss >= current_rss:
            print(f"Deteniendo en el paso {i+1}: No hay mejora significativa.")
            break

        # Añadir la mejor característica
        selected_indices.append(best_index)
        remaining_indices.remove(best_index)
        current_rss = best_rss

        # Calcular R2 con el modelo actual
        model = LinearRegression()
        model.fit(X_array[:, selected_indices], y_array)
        r2 = model.score(X_array[:, selected_indices], y_array)

        history.append({
            'step': i+1,
            'feature_added': feature_names[best_index],
            'rss': current_rss,
            'r2': r2
        })

        print(f"Paso {i+1}: Añadido '{feature_names[best_index]}' | RSS: {current_rss:.2f} | R2: {r2:.4f}")

    return [feature_names[i] for i in selected_indices], pd.DataFrame(history)

# --- Ejecutar Forward Stepwise ---
# Asegúrate de que X_train y y_train estén definidos (de la Fase 1)
feature_names = list(X_train.columns)  # Nombres de las 81 columnas

start_time = time.time()
selected_forward, history_forward = forward_stepwise_selection(
    X_train, y_train, feature_names, max_features=20
)
end_time = time.time()

print(f"\nTiempo de ejecución: {end_time - start_time:.2f} segundos")
print(f"Variables seleccionadas: {selected_forward}")

Paso 1: Añadido 'wtd_std_ThermalConductivity' | RSS: 9652884.16 | R2: 0.5186
Paso 2: Añadido 'wtd_gmean_ElectronAffinity' | RSS: 8885457.94 | R2: 0.5569
Paso 3: Añadido 'range_atomic_radius' | RSS: 8205619.65 | R2: 0.5908
Paso 4: Añadido 'wtd_std_Valence' | RSS: 7922496.28 | R2: 0.6049
Paso 5: Añadido 'wtd_mean_ElectronAffinity' | RSS: 7558852.54 | R2: 0.6230
Paso 6: Añadido 'wtd_std_ElectronAffinity' | RSS: 7345811.85 | R2: 0.6337
Paso 7: Añadido 'wtd_std_atomic_radius' | RSS: 7169417.72 | R2: 0.6424
Paso 8: Añadido 'std_ThermalConductivity' | RSS: 7060182.62 | R2: 0.6479
Paso 9: Añadido 'wtd_std_FusionHeat' | RSS: 6965086.49 | R2: 0.6526
Paso 10: Añadido 'wtd_entropy_ThermalConductivity' | RSS: 6869679.11 | R2: 0.6574
Paso 11: Añadido 'entropy_atomic_mass' | RSS: 6749659.65 | R2: 0.6634
Paso 12: Añadido 'wtd_entropy_atomic_mass' | RSS: 6680421.16 | R2: 0.6668
Paso 13: Añadido 'wtd_range_Valence' | RSS: 6608065.97 | R2: 0.6704
Paso 14: Añadido 'std_atomic_radius' | RSS: 6522108.01 | R

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import time

def backward_stepwise_selection(X, y, min_features=1):
    """
    Realiza Backward Stepwise Selection.
    Devuelve una lista con los predictores finales y el historial.
    """
    # Asegurarnos de que X es un DataFrame y mantener el orden original
    if not isinstance(X, pd.DataFrame):
        X = pd.DataFrame(X)

    # Lista de todas las columnas en el orden original
    all_features = list(X.columns)
    current_features = all_features.copy()

    history = []

    # Ajustar modelo completo inicial
    model = LinearRegression()
    model.fit(X[current_features], y)
    y_pred = model.predict(X[current_features])
    current_rss = mean_squared_error(y, y_pred) * len(y)

    print(f"Modelo completo inicial | RSS: {current_rss:.2f} | R2: {model.score(X[current_features], y):.4f}")

    while len(current_features) > min_features:
        worst_rss = np.inf
        worst_feature = None

        # Probar eliminar cada característica
        for feature in current_features:
            # Crear lista de features sin la actual, manteniendo el orden original
            features_to_test = [f for f in all_features if f in current_features and f != feature]

            # Asegurarnos de que el DataFrame tenga las columnas en el orden correcto
            X_subset = X[features_to_test]

            model = LinearRegression()
            model.fit(X_subset, y)
            y_pred = model.predict(X_subset)
            rss = mean_squared_error(y, y_pred) * len(y)

            if rss < worst_rss:
                worst_rss = rss
                worst_feature = feature

        # Eliminar la peor característica
        current_features.remove(worst_feature)

        # Calcular métricas para el modelo actual (sin la peor feature)
        X_subset = X[current_features]
        model = LinearRegression()
        model.fit(X_subset, y)
        y_pred = model.predict(X_subset)
        rss_after_removal = mean_squared_error(y, y_pred) * len(y)
        r2_after_removal = model.score(X_subset, y)

        history.append({
            'step': len(current_features),
            'feature_removed': worst_feature,
            'rss': rss_after_removal,
            'r2': r2_after_removal
        })

        print(f"Eliminado '{worst_feature}' | Quedan {len(current_features)} variables | RSS: {rss_after_removal:.2f} | R2: {r2_after_removal:.4f}")

        # Detener si el RSS aumenta demasiado (opcional)
        # if rss_after_removal > current_rss * 1.5: # Si el RSS aumenta más del 50%
        #     print("Deteniendo: El RSS ha aumentado significativamente.")
        #     break

        current_rss = rss_after_removal

    return current_features, pd.DataFrame(history)

# Ejecutar Backward Stepwise
start_time = time.time()
selected_backward, history_backward = backward_stepwise_selection(X_train, y_train, min_features=1)
end_time = time.time()

print(f"\nTiempo de ejecución: {end_time - start_time:.2f} segundos")
print(f"Variables finales: {selected_backward}")

Modelo completo inicial | RSS: 5276808.16 | R2: 0.7368
Eliminado 'gmean_atomic_radius' | Quedan 80 variables | RSS: 5276811.43 | R2: 0.7368
Eliminado 'wtd_entropy_ThermalConductivity' | Quedan 79 variables | RSS: 5276889.13 | R2: 0.7368
Eliminado 'wtd_mean_Density' | Quedan 78 variables | RSS: 5277025.82 | R2: 0.7368
Eliminado 'wtd_range_Density' | Quedan 77 variables | RSS: 5277136.43 | R2: 0.7368
Eliminado 'wtd_mean_fie' | Quedan 76 variables | RSS: 5277293.70 | R2: 0.7368
Eliminado 'wtd_entropy_atomic_mass' | Quedan 75 variables | RSS: 5277592.29 | R2: 0.7368
Eliminado 'wtd_range_atomic_mass' | Quedan 74 variables | RSS: 5277893.07 | R2: 0.7368
Eliminado 'wtd_range_Valence' | Quedan 73 variables | RSS: 5278139.50 | R2: 0.7368
Eliminado 'gmean_fie' | Quedan 72 variables | RSS: 5278658.09 | R2: 0.7367
Eliminado 'mean_fie' | Quedan 71 variables | RSS: 5278734.47 | R2: 0.7367
Eliminado 'wtd_std_atomic_mass' | Quedan 70 variables | RSS: 5279426.77 | R2: 0.7367
Eliminado 'wtd_std_ThermalC

In [ ]:
from sklearn.model_selection import cross_val_score

def calculate_bic(X, y, features):
    """Calcula el BIC para un conjunto de features."""
    model = LinearRegression()
    model.fit(X[features], y)
    y_pred = model.predict(X[features])
    rss = mean_squared_error(y, y_pred) * len(y)
    n = len(y)
    k = len(features) + 1 # +1 por el intercepto
    bic = n * np.log(rss/n) + k * np.log(n)
    return bic

def evaluate_model_cv(X, y, features, cv=5):
    """Calcula el MSE de validación cruzada."""
    model = LinearRegression()
    scores = cross_val_score(model, X[features], y, cv=cv, scoring='neg_mean_squared_error')
    return -scores.mean()

# Evaluar Forward Stepwise (usando la historia)
print("Evaluando Forward Stepwise...")
forward_results = []
for i in range(1, len(selected_forward) + 1):
    features = selected_forward[:i]
    bic = calculate_bic(X_train, y_train, features)
    cv_mse = evaluate_model_cv(X_train, y_train, features)
    forward_results.append({'num_features': i, 'features': features, 'bic': bic, 'cv_mse': cv_mse})
    print(f"Paso {i}: {i} variables | BIC: {bic:.2f} | CV MSE: {cv_mse:.2f}")

# Evaluar Backward Stepwise (usando la historia)
# NOTA: La historia de backward guarda las variables que QUEDAN, no las que se eliminan.
# Necesitamos reconstruir los conjuntos de variables.
print("\nEvaluando Backward Stepwise...")
backward_results = []
# Empezamos desde el modelo completo y vamos eliminando según la historia
current_features = list(X_train.columns)
# La historia tiene los pasos de eliminación. El primer paso elimina 1, quedan 80.
# Reconstruimos los conjuntos de variables para cada tamaño.
# (Esto es un poco complejo, pero aquí está la lógica)
# Para simplificar, evaluaremos el modelo completo y luego los que quedan tras cada eliminación.
# Pero para el informe, basta con evaluar el modelo final y algunos intermedios.
# Haremos una simplificación: evaluar el modelo completo y el final.
print(f"Modelo Completo: BIC = {calculate_bic(X_train, y_train, list(X_train.columns)):.2f}, CV MSE = {evaluate_model_cv(X_train, y_train, list(X_train.columns)):.2f}")
print(f"Modelo Final Backward: BIC = {calculate_bic(X_train, y_train, selected_backward):.2f}, CV MSE = {evaluate_model_cv(X_train, y_train, selected_backward):.2f}")

# Para una comparación justa, necesitamos evaluar todos los tamaños.
# Aquí hay un código para hacerlo correctamente:
backward_results = []
current_features = list(X_train.columns)
# Evaluar modelo completo
bic_full = calculate_bic(X_train, y_train, current_features)
cv_full = evaluate_model_cv(X_train, y_train, current_features)
backward_results.append({'num_features': len(current_features), 'bic': bic_full, 'cv_mse': cv_full})

for index, row in history_backward.iterrows():
    current_features.remove(row['feature_removed'])
    bic = calculate_bic(X_train, y_train, current_features)
    cv_mse = evaluate_model_cv(X_train, y_train, current_features)
    backward_results.append({'num_features': len(current_features), 'bic': bic, 'cv_mse': cv_mse})
    # print(f"Quedan {len(current_features)} variables | BIC: {bic:.2f} | CV MSE: {cv_mse:.2f}")

# Encontrar el mejor modelo según BIC y CV para ambos métodos
best_forward_bic = min(forward_results, key=lambda x: x['bic'])
best_forward_cv = min(forward_results, key=lambda x: x['cv_mse'])
best_backward_bic = min(backward_results, key=lambda x: x['bic'])
best_backward_cv = min(backward_results, key=lambda x: x['cv_mse'])

print("\n--- MEJORES MODELOS ---")
print(f"Forward - Mejor BIC: {best_forward_bic['num_features']} variables, BIC={best_forward_bic['bic']:.2f}")
print(f"Forward - Mejor CV: {best_forward_cv['num_features']} variables, CV MSE={best_forward_cv['cv_mse']:.2f}")
print(f"Backward - Mejor BIC: {best_backward_bic['num_features']} variables, BIC={best_backward_bic['bic']:.2f}")
print(f"Backward - Mejor CV: {best_backward_cv['num_features']} variables, CV MSE={best_backward_cv['cv_mse']:.2f}")

Evaluando Forward Stepwise...
Paso 1: 1 variables | BIC: 107883.48 | CV MSE: 567.85
Paso 2: 2 variables | BIC: 106484.10 | CV MSE: 522.72
Paso 3: 3 variables | BIC: 105139.90 | CV MSE: 482.74
Paso 4: 4 variables | BIC: 104552.37 | CV MSE: 466.12
Paso 5: 5 variables | BIC: 103762.86 | CV MSE: 444.73
Paso 6: 6 variables | BIC: 103286.30 | CV MSE: 432.26
Paso 7: 7 variables | BIC: 102882.60 | CV MSE: 421.98
Paso 8: 8 variables | BIC: 102631.18 | CV MSE: 415.59
Paso 9: 9 variables | BIC: 102410.25 | CV MSE: 409.99
Paso 10: 10 variables | BIC: 102185.38 | CV MSE: 404.40
Paso 11: 11 variables | BIC: 101895.31 | CV MSE: 397.39
Paso 12: 12 variables | BIC: 101729.66 | CV MSE: 393.33
Paso 13: 13 variables | BIC: 101554.17 | CV MSE: 389.10
Paso 14: 14 variables | BIC: 101341.19 | CV MSE: 384.10
Paso 15: 15 variables | BIC: 101117.79 | CV MSE: 378.91
Paso 16: 16 variables | BIC: 101023.05 | CV MSE: 376.59
Paso 17: 17 variables | BIC: 100809.72 | CV MSE: 371.78
Paso 18: 18 variables | BIC: 100672.